In [1]:
import math

# 1. GEOMETRY
# Horizontal distances from Support A (x=0)
x_A = 0.0
x_E = 1.5
x_B = 3.0
x_F = 4.5
x_C = 6.0
x_G = 7.5
x_D = 9.0

# Vertical depth of the truss
h = 1.5

# 2. EXTERNAL LOADS (in kN)
# Downward loads on the bottom chord
F_Ey = -60.0
F_Fy = -200.0
F_Gy = -100.0

# Angled 200 kN loads at B and C (70 degrees)
# Pointing downwards and leftwards
P_skew = 200.0
angle_rad = math.radians(70)

# Decomposing into X and Y components
F_Bx = -P_skew * math.cos(angle_rad)
F_By = -P_skew * math.sin(angle_rad)

F_Cx = -P_skew * math.cos(angle_rad)
F_Cy = -P_skew * math.sin(angle_rad)

print("--- EXTERNAL LOADS CALCULATED ---")
print(f"Node B -> Fx: {F_Bx:.2f} kN | Fy: {F_By:.2f} kN")
print(f"Node C -> Fx: {F_Cx:.2f} kN | Fy: {F_Cy:.2f} kN")

--- EXTERNAL LOADS CALCULATED ---
Node B -> Fx: -68.40 kN | Fy: -187.94 kN
Node C -> Fx: -68.40 kN | Fy: -187.94 kN


In [2]:
# 3. GLOBAL REACTIONS
# To find R_Ay, we take the sum of moments around Support D (x = 9.0)
# Counter-clockwise moments are considered positive.
# Lever arm is the horizontal distance from the load to Support D: (x_D - x_load)

# Sum of Moments about D = 0
moment_E = abs(F_Ey) * (x_D - x_E)
moment_B = abs(F_By) * (x_D - x_B)
moment_F = abs(F_Fy) * (x_D - x_F)
moment_C = abs(F_Cy) * (x_D - x_C)
moment_G = abs(F_Gy) * (x_D - x_G)

# The horizontal forces at B and C act along the top chord (y=0),
# which passes directly through D (y=0), so they create zero moment.
total_CCW_moment = moment_E + moment_B + moment_F + moment_C + moment_G

# R_Ay creates a clockwise (negative) moment with a lever arm of 9.0m
# R_Ay * 9.0 = total_CCW_moment
R_Ay = total_CCW_moment / x_D

# Sum of forces in X = 0
# The pinned support A takes all the horizontal thrust
R_Ax = -(F_Bx + F_Cx)

# Sum of forces in Y = 0 to find the roller reaction at D (R_Dy)
# Upward forces + Downward forces = 0
total_downward_Fy = F_Ey + F_By + F_Fy + F_Cy + F_Gy
R_Dy = -total_downward_Fy - R_Ay

print("--- GLOBAL SUPPORT REACTIONS ---")
print(f"Reaction at A (X-dir): R_Ax = {R_Ax:.2f} kN")
print(f"Reaction at A (Y-dir): R_Ay = {R_Ay:.2f} kN")
print(f"Reaction at D (Y-dir): R_Dy = {R_Dy:.2f} kN")

# Quick equilibrium check
equilibrium_check = R_Ay + R_Dy + total_downward_Fy
print(f"Vertical Equilibrium Check (should be 0): {equilibrium_check:.4f}")

--- GLOBAL SUPPORT REACTIONS ---
Reaction at A (X-dir): R_Ax = 136.81 kN
Reaction at A (Y-dir): R_Ay = 354.61 kN
Reaction at D (Y-dir): R_Dy = 381.27 kN
Vertical Equilibrium Check (should be 0): 0.0000


In [3]:
# 4. METHOD OF SECTIONS (Calculating Force in Bar EF)
# We make a vertical cut through bars BC, BF, and EF.
# We analyze the left portion of the truss and sum moments about Node B.
# We assume N_EF is in TENSION (pulling to the right from Node E).

# Calculate lever arms relative to Node B (x=3.0, y=0.0)
lever_arm_RAy = x_B - x_A  # 3.0m (horizontal distance from A to B)
lever_arm_FEy = x_B - x_E  # 1.5m (horizontal distance from E to B)
lever_arm_NEF = h          # 1.5m (vertical distance from E to B)

# Sum of Moments about Node B = 0 (Counter-Clockwise is Positive)
# 1. Reaction A pushes UP from the left -> Creates Clockwise moment (-)
moment_RAy = -R_Ay * lever_arm_RAy

# 2. Load at E pulls DOWN from the left -> Creates Counter-Clockwise moment (+)
moment_FEy = abs(F_Ey) * lever_arm_FEy

# 3. Force N_EF pulls RIGHT from the bottom -> Creates Counter-Clockwise moment (+)
# The equation is: moment_RAy + moment_FEy + (N_EF * lever_arm_NEF) = 0

# Solving the equation algebraically for N_EF:
N_EF = -(moment_RAy + moment_FEy) / lever_arm_NEF

print("--- INTERNAL FORCES (METHOD OF SECTIONS) ---")
print(f"Axial Force in Bar EF (N_Ed): {N_EF:.2f} kN")

if N_EF > 0:
    print("Status: TENSION (Tracción)")
else:
    print("Status: COMPRESSION (Compresión)")

--- INTERNAL FORCES (METHOD OF SECTIONS) ---
Axial Force in Bar EF (N_Ed): 649.21 kN
Status: TENSION (Tracción)


In [ ]:
# --- TEMA 4: DIMENSIONADO DEL PERFIL (TRACCIÓN) ---

# 1. MATERIAL PROPERTIES & FACTORS
N_Ed_kN = 649.22        # Design axial force in kN (from Cell 3)
N_Ed_N = N_Ed_kN * 1000 # Convert to Newtons for consistent units
f_y = 275.0             # Yield strength of S-275 steel in MPa (N/mm^2)
gamma_M0 = 1.05         # Partial safety factor for steel yielding (Ys)

# 2. REQUIRED AREA CALCULATION
# Design yield strength (f_yd)
f_yd = f_y / gamma_M0

# Required gross area (A_req) in mm^2, then converted to cm^2
A_req_mm2 = N_Ed_N / f_yd
A_req_cm2 = A_req_mm2 / 100

# Since we are using a DOUBLE UPN, each single UPN needs half the area
A_req_single_cm2 = A_req_cm2 / 2

print("--- CALCULO DE ÁREA REQUERIDA ---")
print(f"Esfuerzo de diseño (N_Ed): {N_Ed_kN:.2f} kN")
print(f"Resistencia de cálculo (f_yd): {f_yd:.2f} N/mm2")
print(f"Área total requerida: {A_req_cm2:.2f} cm2")
print(f"Área requerida por cada perfil UPN: {A_req_single_cm2:.2f} cm2\n")

# 3. PROFILE CATALOG (Prontuario)
# Dictionary mapping UPN profile names to their Gross Area (A in cm2)
# You can expand this list based on your specific prontuario
upn_catalog = {
    "UPN 80": 11.0,
    "UPN 100": 13.5,
    "UPN 120": 17.0,
    "UPN 140": 20.4,
    "UPN 160": 24.0,
    "UPN 180": 28.0,
    "UPN 200": 32.2
}

# 4. SELECT THE OPTIMAL PROFILE
selected_profile = None
selected_area = 0.0

for profile, area in upn_catalog.items():
    if area >= A_req_single_cm2:
        selected_profile = profile
        selected_area = area
        break # Stop at the first profile that is large enough

print("--- SELECCIÓN DE PERFIL ---")
if selected_profile:
    print(f"Perfil seleccionado: DOBLE {selected_profile}")
    print(f"Área aportada por un {selected_profile}: {selected_area:.2f} cm2")
    print(f"Área total aportada (2x): {selected_area * 2:.2f} cm2 > {A_req_cm2:.2f} cm2 (OK!)")
else:
    print("Error: Ningún perfil en el catálogo es suficientemente grande.")

In [4]:
# --- TEMA 4: DIMENSIONADO DEL PERFIL (TRACCIÓN) ---

# 1. MATERIAL PROPERTIES & FACTORS
N_Ed_kN = 649.22        # Design axial force in kN (from Cell 3)
N_Ed_N = N_Ed_kN * 1000 # Convert to Newtons for consistent units
f_y = 275.0             # Yield strength of S-275 steel in MPa (N/mm^2)
gamma_M0 = 1.05         # Partial safety factor for steel yielding (Ys)

# 2. REQUIRED AREA CALCULATION
# Design yield strength (f_yd)
f_yd = f_y / gamma_M0

# Required gross area (A_req) in mm^2, then converted to cm^2
A_req_mm2 = N_Ed_N / f_yd
A_req_cm2 = A_req_mm2 / 100

# Since we are using a DOUBLE UPN, each single UPN needs half the area
A_req_single_cm2 = A_req_cm2 / 2

print("--- CALCULO DE ÁREA REQUERIDA ---")
print(f"Esfuerzo de diseño (N_Ed): {N_Ed_kN:.2f} kN")
print(f"Resistencia de cálculo (f_yd): {f_yd:.2f} N/mm2")
print(f"Área total requerida: {A_req_cm2:.2f} cm2")
print(f"Área requerida por cada perfil UPN: {A_req_single_cm2:.2f} cm2\n")

# 3. PROFILE CATALOG (Prontuario)
# Dictionary mapping UPN profile names to their Gross Area (A in cm2)
# You can expand this list based on your specific prontuario
upn_catalog = {
    "UPN 80": 11.0,
    "UPN 100": 13.5,
    "UPN 120": 17.0,
    "UPN 140": 20.4,
    "UPN 160": 24.0,
    "UPN 180": 28.0,
    "UPN 200": 32.2
}

# 4. SELECT THE OPTIMAL PROFILE
selected_profile = None
selected_area = 0.0

for profile, area in upn_catalog.items():
    if area >= A_req_single_cm2:
        selected_profile = profile
        selected_area = area
        break # Stop at the first profile that is large enough

print("--- SELECCIÓN DE PERFIL ---")
if selected_profile:
    print(f"Perfil seleccionado: DOBLE {selected_profile}")
    print(f"Área aportada por un {selected_profile}: {selected_area:.2f} cm2")
    print(f"Área total aportada (2x): {selected_area * 2:.2f} cm2 > {A_req_cm2:.2f} cm2 (OK!)")
else:
    print("Error: Ningún perfil en el catálogo es suficientemente grande.")

--- CALCULO DE ÁREA REQUERIDA ---
Esfuerzo de diseño (N_Ed): 649.22 kN
Resistencia de cálculo (f_yd): 261.90 N/mm2
Área total requerida: 24.79 cm2
Área requerida por cada perfil UPN: 12.39 cm2

--- SELECCIÓN DE PERFIL ---
Perfil seleccionado: DOBLE UPN 100
Área aportada por un UPN 100: 13.50 cm2
Área total aportada (2x): 27.00 cm2 > 24.79 cm2 (OK!)
